# Week 2: Geospatial Prompting and Validation

This notebook contains **Lab 2**, a 30-minute paired in-class exercise worth 4% of the final grade, and **Homework 2**, a take-home exercise.

Tools: [Tiktokenizer](https://tiktokenizer.vercel.app/), an LLM, [geojson.io](https://geojson.io), and Google Colab. Homework also uses [Copernicus Browser](https://browser.dataspace.copernicus.eu/), [NASA Earthdata Search](https://search.earthdata.nasa.gov/), and [Kaggle Datasets](https://www.kaggle.com/datasets).

**R-C-S-P-V** means Role, CRS, Schema, Physics, and Verify.

In [1]:
%pip install -q tiktoken geopandas shapely leafmap 'pydantic>=2'

Note: you may need to restart the kernel to use updated packages.


ERROR: Invalid requirement: "'pydantic": Expected package name at the start of dependency specifier
    'pydantic
    ^


In [2]:
import json
from datetime import date
from pathlib import Path
from typing import Literal, Optional

import geopandas as gpd
import leafmap
import tiktoken
from pydantic import BaseModel, ConfigDict, Field

enc = tiktoken.get_encoding('o200k_base')

# Part 1 — Lab 2: Prompt Golf

**Duration:** 30 minutes, worked in pairs. **Weight:** 4% of the final course grade, assessed through the in-class micro-audit worksheet.

1. Open four tabs: Tiktokenizer, your preferred LLM, geojson.io, and this Colab notebook. Select `o200k_base` or `cl100k_base` in Tiktokenizer.
2. Inspect the six-record log below. Identify DMS coordinates, Record 04's longitude-first notation, Record 02's NoData sentinel, Record 06's impossible cloud cover, and mixed dates.
3. Write an R-C-S-P-V prompt of **at most 135 tokens**, excluding the raw data. Birdie: at most 120 tokens. Record the tokenizer used.
4. Give the prompt and log to your LLM. Paste its raw JSON into geojson.io and inspect all six points.
5. Paste the prompt and output into the cells below. Run the checks and complete the micro-audit.

**Note:** GeoJSON coordinates are `[longitude, latitude]`. Record 04 is already written longitude first in the source; preserve the labeled values in that order.

## 1.1 Raw input log

Copy this block below your prompt:

```text
--- RAW MULTI-SENSOR EO & GEOMATICS CAMPAIGN LOG (ALBORZ / TEHRAN BASIN) ---
Record_01 [Copernicus Browser - RS]: Granule=S2B_MSIL2A_20250714T071619_T39STT | Acq_Date: 14-Jul-2025 | Center_DMS: 35°45'00"N, 51°30'00"E | Cloud_Cover: 12.4% | Mean_NDVI: 0.642 | QA_Status: Nominal
Record_02 [NASA Earthdata - RS Thermal]: Granule=MOD11A1_A2025196 | Date: 2025/07/15 | Center: Lat 35.7000, Lon 51.4000 | LST_Celsius: -9999 (Sensor_NoData_Flag) | Cloud_Cover: 85.0% | QA_Status: Masked_NoData
Record_03 [NASA Earthdata GEDI & UAV - Photogrammetry/LiDAR]: Shot_ID=GEDI_L2A_99412 | Date: July 16, 2025 | Pos: 35°48'45"N, 51°25'30"E | Canopy_Height_rh98: 18.6m | Cloud_Cover: N/A | QA_Status: Valid_Waveform
Record_04 [Geodetic Benchmark - Geodesy]: Station=BM_TEH_04 (Trimble R10 Static GNSS) | Logged: 2025-07-16 | Coords_Reversed: 51°21'00"E, 35°42'00"N (WARNING: Field crew wrote Longitude first!) | Ortho_Height_m: 1420.55 | Cloud_Cover: N/A | QA_Status: Fixed_Solution
Record_05 [Kaggle Agro-GIS Soil Probe - GIS]: Probe_ID=KAG_SOIL_08 | Date: 17.07.2025 | Location: 35.6500N, 51.3500E | Soil_pH: 7.42 | Cloud_Cover: 0.0% | QA_Status: Verified_Field_Sample
Record_06 [Corrupted Sentinel-2 Metadata - RS Trap]: Granule=S2A_MSIL2A_20250718_T39STT | Date: 2025-07-18 | Center: 35°51'00"N, 51°33'00"E | Cloud_Cover: 145.0% (Impossible_Corrupted_Header) | Mean_NDVI: 0.210 | QA_Status: Header_Error
```

## 1.2 Instructor reference prompt

Measure the actual token count with your selected encoding; counts can differ by tokenizer. Temperature is an LLM setting when available, not something a prompt can guarantee.

```text
ROLE: Deterministic GeoJSON parser (Temp=0). Output raw valid GeoJSON FeatureCollection only; no markdown prose.
CRS: EPSG:4326. Geometry: Point with coordinates strictly [longitude, latitude] decimal degrees (round to 4 decimals; convert DMS; fix reversed Lon/Lat).
SCHEMA per Feature properties:
- "id": string (Record_01 to Record_06)
- "major": "RS" | "GIS" | "PHOTO" | "GEODESY"
- "date_iso": "YYYY-MM-DD"
- "cloud_pct": float or null (set N/A or >100 to null)
- "primary_metric": float or null (set -9999 to null)
- "qa_valid": boolean (false if NoData or Header_Error)
DATA:
[PASTE RAW 6 RECORDS HERE]
```

In [8]:
# Lab token audit: replace this with your instructions, excluding the raw log.
student_prompt = '''R: Deterministic GeoJSON parser, T=0.
C: Output only a raw FeatureCollection, no prose. EPSG:4326.
S: Point [lon,lat], 4 decimals; convert DMS; fix swapped lon/lat.
P: id Record_01..06; major RS|GIS|PHOTO|GEODESY; date_iso YYYY-MM-DD; cloud_pct float, null if N/A or >100; primary_metric float, null if -9999; qa_valid false if NoData or Header_Error.
V: Valid JSON only.'''

dms_str = '''Station_01: Lat 35°41'21.48"N, Lon 51°23'20.52"E, Elev: 1,189.40m'''
dec_str = '["S01", 35.6893, 51.3890, 1189.4]'
print('DMS tokens:', len(enc.encode(dms_str)))
print('Decimal tokens:', len(enc.encode(dec_str)))
prompt_tokens = len(enc.encode(student_prompt))
print('Prompt tokens:', prompt_tokens, '| Par: <= 135 | Birdie: <= 120')
assert student_prompt != 'PASTE YOUR R-C-S-P-V PROMPT HERE', 'Paste your prompt first.'
assert prompt_tokens <= 135, 'Over par: shorten your instructions.'

DMS tokens: 35
Decimal tokens: 22
Prompt tokens: 117 | Par: <= 135 | Birdie: <= 120


In [11]:
# Paste your LLM's raw GeoJSON between the triple quotes.
lab_geojson_str = '''{"type":"FeatureCollection","features":[{"type":"Feature","geometry":{"type":"Point","coordinates":[51.5000,35.7500]},"properties":{"id":"Record_01","major":"RS","date_iso":"2025-07-14","cloud_pct":12.4,"primary_metric":0.642,"qa_valid":true}},{"type":"Feature","geometry":{"type":"Point","coordinates":[51.4000,35.7000]},"properties":{"id":"Record_02","major":"RS","date_iso":"2025-07-15","cloud_pct":85.0,"primary_metric":null,"qa_valid":false}},{"type":"Feature","geometry":{"type":"Point","coordinates":[51.4250,35.8125]},"properties":{"id":"Record_03","major":"PHOTO","date_iso":"2025-07-16","cloud_pct":null,"primary_metric":18.6,"qa_valid":true}},{"type":"Feature","geometry":{"type":"Point","coordinates":[51.3500,35.7000]},"properties":{"id":"Record_04","major":"GEODESY","date_iso":"2025-07-16","cloud_pct":null,"primary_metric":1420.55,"qa_valid":true}},{"type":"Feature","geometry":{"type":"Point","coordinates":[51.3500,35.6500]},"properties":{"id":"Record_05","major":"GIS","date_iso":"2025-07-17","cloud_pct":0.0,"primary_metric":7.42,"qa_valid":true}},{"type":"Feature","geometry":{"type":"Point","coordinates":[51.5500,35.8500]},"properties":{"id":"Record_06","major":"RS","date_iso":"2025-07-18","cloud_pct":null,"primary_metric":0.210,"qa_valid":false}}]}'''

def load_strict_json(text):
    def reject_constant(value):
        raise ValueError(f'Nonstandard JSON constant: {value}')
    return json.loads(text, parse_constant=reject_constant)

def validate_lab(text):
    data = load_strict_json(text)
    assert data.get('type') == 'FeatureCollection', 'Expected a FeatureCollection.'
    features = data.get('features', [])
    assert len(features) == 6, f'Expected 6 features, got {len(features)}.'
    by_id = {f['properties']['id']: f for f in features}
    assert len(by_id) == 6 and set(by_id) == {f'Record_0{i}' for i in range(1, 7)}, 'Missing or duplicate IDs.'
    for record_id, feature in by_id.items():
        assert feature.get('type') == 'Feature'
        geom = feature['geometry']
        assert geom['type'] == 'Point' and len(geom['coordinates']) == 2
        lon, lat = geom['coordinates']
        assert 51.0 < lon < 52.0 and 35.0 < lat < 36.0, f'{record_id}: check coordinate order and location.'
        props = feature['properties']
        assert props['major'] in ('RS', 'GIS', 'PHOTO', 'GEODESY')
        date.fromisoformat(props['date_iso'])
        cloud = props['cloud_pct']
        assert cloud is None or (not isinstance(cloud, bool) and 0 <= cloud <= 100)
        assert isinstance(props['qa_valid'], bool)
    assert [round(x, 4) for x in by_id['Record_03']['geometry']['coordinates']] == [51.425, 35.8125], 'Record 03 DMS error.'
    assert [round(x, 4) for x in by_id['Record_04']['geometry']['coordinates']] == [51.35, 35.7], 'Record 04 axis error.'
    assert by_id['Record_02']['properties']['primary_metric'] is None, 'Record 02 NoData error.'
    assert by_id['Record_06']['properties']['cloud_pct'] is None, 'Record 06 cloud-cover error.'
    assert not by_id['Record_02']['properties']['qa_valid'] and not by_id['Record_06']['properties']['qa_valid'], 'Invalid QA flags.'
    return data

lab_data = validate_lab(lab_geojson_str)
Path('lab2_verified.geojson').write_text(json.dumps(lab_data, indent=2), encoding='utf-8')
print('Lab checks passed. Saved lab2_verified.geojson.')

Lab checks passed. Saved lab2_verified.geojson.


In [12]:
# Run after the lab validation cell.
m = leafmap.Map(center=[35.75, 51.42], zoom=10)
m.add_basemap('Esri.WorldImagery')
m.add_geojson('lab2_verified.geojson', layer_name='Verified campaign')
m

Map(center=[35.75, 51.42], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_o…

## 1.3 Micro-audit worksheet

- Pair members: ____________________
- Tokenizer and prompt token count: ____________________
- DMS versus decimal token counts: ____________________
- Did all six points appear in the expected region? ____________________
- How were Record 02 and Record 06 handled? ____________________
- What did you change after validation? ____________________

# Part 2 — Homework 2: Portal-to-Pipeline

Build an eight-record messy multi-sensor log, parse it into GeoJSON, inspect it visually, and validate it in Colab.

## Instructions

1. **Harvest four real portal records** for a study area of your choice: two Copernicus scenes, one NASA Earthdata granule, and one Kaggle environmental or soil row. Record source links and metadata. Assign the Kaggle row a valid study-area coordinate if needed, and label that coordinate as assigned.
2. **Add four synthetic field records**, covering these traps: verbose DMS; a reversed or mislabeled GNSS axis; a `-9999` or `-32768` NoData sentinel; and a physical violation such as cloud cover above 100% or temperature below 0 K. Label synthetic records clearly.
3. **Audit token compression:** compare an eight-decimal WKT polygon or verbose DMS string with a compact version or coordinates rounded to five decimals. Record token counts and percentage savings. Five decimal places of latitude are approximately 1.1 m; longitude spacing varies with latitude.
4. **Compare prompts:** run the log through an LLM with both `Please convert this text into a GeoJSON file` and an R-C-S-P-V prompt of at most 140 tokens. Inspect both outputs in geojson.io and capture a comparison screenshot.
5. **Validate and submit:** run the cells below on your structured-prompt output. Export the notebook and validated `.geojson` file and submit them to your repository, along with the requested screenshot.

Record your sources, raw log, prompts, both outputs, model name, token counts, and observations in the cells below.

## 2.1 Research notes

- Student name(s): ____________________
- Study area: ____________________
- LLM/model: ____________________
- Source links and provenance: ____________________
- Assigned coordinate and rationale: ____________________
- Synthetic trap descriptions: ____________________
- Comparison observations and screenshot path: ____________________

**Paste your eight-record log and both LLM outputs here, or document their repository paths.**

In [ ]:
# Homework token audit: replace the examples with your own strings and prompt.
raw_spatial_string = 'POLYGON ((51.38901234 35.68929876, 51.39104567 35.68929876, 51.39104567 35.69101122, 51.38901234 35.68929876))'
compact_spatial_string = '[[51.38901,35.68930],[51.39105,35.68930],[51.39105,35.69101],[51.38901,35.68930]]'
homework_prompt = '''PASTE YOUR R-C-S-P-V PROMPT HERE'''

raw_tok = len(enc.encode(raw_spatial_string))
comp_tok = len(enc.encode(compact_spatial_string))
assert raw_tok > 0
print(f'Spatial tokens: {raw_tok} -> {comp_tok}; savings: {(raw_tok - comp_tok) / raw_tok * 100:.1f}%')
hw_prompt_tokens = len(enc.encode(homework_prompt))
print(f'R-C-S-P-V prompt: {hw_prompt_tokens} tokens; target <= 140')
assert homework_prompt != 'PASTE YOUR R-C-S-P-V PROMPT HERE', 'Paste your prompt first.'
assert hw_prompt_tokens <= 140, 'Shorten the homework prompt.'

## 2.2 Homework validator

For each feature, use properties `id`, `portal_source` (`Copernicus`, `Earthdata`, `Kaggle`, or `Field_Survey`), `date_iso`, `cloud_pct`, `primary_metric`, and `qa_valid`. Use `null` for missing or invalid values and `false` for invalid QA.

The validator checks structure and selected guardrails; it does **not** prove that portal metadata, trap corrections, source provenance, or point locations are accurate. Compare those manually with your log.

In [ ]:
class FeatureProperties(BaseModel):
    model_config = ConfigDict(strict=True, extra='allow')
    id: str
    portal_source: Literal['Copernicus', 'Earthdata', 'Kaggle', 'Field_Survey']
    date_iso: str = Field(pattern=r'^\d{4}-\d{2}-\d{2}$')
    cloud_pct: Optional[float] = Field(ge=0.0, le=100.0)
    primary_metric: Optional[float]
    qa_valid: bool

def grade_homework_geojson(text, lon_bounds, lat_bounds):
    data = load_strict_json(text)
    assert data.get('type') == 'FeatureCollection', 'Expected a FeatureCollection.'
    features = data.get('features', [])
    assert len(features) == 8, f'Expected 8 features, found {len(features)}.'
    ids = set()
    for idx, feat in enumerate(features, start=1):
        assert feat.get('type') == 'Feature', f'Record {idx}: expected Feature.'
        props = FeatureProperties.model_validate(feat['properties'])
        date.fromisoformat(props.date_iso)
        assert props.id not in ids, f'Duplicate ID: {props.id}'
        ids.add(props.id)
        assert props.primary_metric not in (-9999, -32768), f'{props.id}: NoData sentinel retained.'
        geom = feat['geometry']
        assert geom['type'] == 'Point' and len(geom['coordinates']) == 2, f'{props.id}: expected a 2D Point.'
        lon, lat = geom['coordinates']
        assert all(isinstance(x, (int, float)) and not isinstance(x, bool) for x in (lon, lat))
        assert -180 <= lon <= 180 and -90 <= lat <= 90, f'{props.id}: invalid geographic coordinates.'
        assert lon_bounds[0] <= lon <= lon_bounds[1] and lat_bounds[0] <= lat <= lat_bounds[1], f'{props.id}: outside study bounds; check axis order.'
    gdf = gpd.GeoDataFrame.from_features(features, crs='EPSG:4326')
    assert gdf.geometry.is_valid.all(), 'Invalid geometry.'
    print('Homework structural checks passed. Verify source values and traps manually.')
    return data, gdf[['id', 'portal_source', 'date_iso', 'cloud_pct', 'primary_metric', 'qa_valid', 'geometry']]

In [ ]:
# Replace the example bounds with your own study-area bounds.
STUDY_LON_BOUNDS = (44.0, 63.5)
STUDY_LAT_BOUNDS = (25.0, 40.0)

# Paste the structured-prompt output here.
homework_geojson_str = '''
PASTE GEOJSON HERE
'''

homework_data, homework_summary = grade_homework_geojson(
    homework_geojson_str, STUDY_LON_BOUNDS, STUDY_LAT_BOUNDS
)
Path('homework2_verified.geojson').write_text(json.dumps(homework_data, indent=2), encoding='utf-8')
print('Saved homework2_verified.geojson.')
homework_summary

## Submission checklist

- [ ] Notebook includes source links, log, prompts, token audit, outputs, and observations.
- [ ] Both outputs inspected in geojson.io; comparison screenshot saved.
- [ ] Structured output checked against the original records and synthetic traps.
- [ ] `homework2_verified.geojson` exported and submitted with the notebook and screenshot.

In [3]:
# ==============================================================================
# WEEK 02 IN-CLASS LAB 2: FULL GOOGLE COLAB VERIFICATION & MAPPING SUITE
# Tools integrated: tiktoken (Tokenizer), JSON/GeoPandas Guardrails, Leafmap
# ==============================================================================
# !pip install -q tiktoken geopandas shapely leafmap

import json
import tiktoken
import geopandas as gpd
import leafmap

# ------------------------------------------------------------------------------
# PART A: AUTOMATED TIKTOKENIZER AUDIT (DMS BLOAT & PROMPT GOLF CHECK)
# ------------------------------------------------------------------------------
enc = tiktoken.get_encoding("o200k_base")

dms_str = 'Station_01: Lat 35°41\'21.48"N, Lon 51°23\'20.52"E, Elev: 1,189.40m'
dec_str = '["S01", 35.6893, 51.3890, 1189.4]'

print("=== 1. TIKTOKENIZER COORDINATE BLOAT EXPERIMENT ===")
print(f"Verbose DMS String Tokens : {len(enc.encode(dms_str))} tokens")
print(f"Compact Decimal JSON Tokens: {len(enc.encode(dec_str))} tokens\n")

# Paste your R-C-S-P-V instructions (WITHOUT the raw data) to verify Par <= 135:
student_prompt = """ROLE: Deterministic GeoJSON parser (Temp=0). Output raw valid GeoJSON FeatureCollection only; no markdown prose.
CRS: EPSG:4326. Geometry: Point with coordinates strictly [longitude, latitude] decimal degrees (round to 4 decimals; convert DMS; fix reversed Lon/Lat).
SCHEMA per Feature properties:
- "id": string (Record_01 to Record_06)
- "major": "RS" | "GIS" | "PHOTO" | "GEODESY"
- "date_iso": "YYYY-MM-DD"
- "cloud_pct": float or null (set N/A or >100 to null)
- "primary_metric": float or null (set -9999 to null)
- "qa_valid": boolean (false if NoData or Header_Error)
DATA:"""

prompt_tokens = len(enc.encode(student_prompt))
print(f"=== 2. PROMPT GOLF SCORE: {prompt_tokens} Tokens (Target Par <= 135) ===")
assert prompt_tokens <= 135, f"Over Par! Reduce your prompt by {prompt_tokens - 135} tokens."

# ------------------------------------------------------------------------------
# PART B: DETERMINISTIC GEOJSON & PHYSICAL GUARDRAIL VALIDATOR
# ------------------------------------------------------------------------------
# Paste the raw GeoJSON output from your LLM inside the triple quotes below:
llm_geojson_str = '''
{
  "type": "FeatureCollection",
  "features": [
    {"type": "Feature", "geometry": {"type": "Point", "coordinates": [51.5, 35.75]}, "properties": {"id": "Record_01", "major": "RS", "date_iso": "2025-07-14", "cloud_pct": 12.4, "primary_metric": 0.642, "qa_valid": true}},
    {"type": "Feature", "geometry": {"type": "Point", "coordinates": [51.4, 35.7]}, "properties": {"id": "Record_02", "major": "RS", "date_iso": "2025-07-15", "cloud_pct": 85.0, "primary_metric": null, "qa_valid": false}},
    {"type": "Feature", "geometry": {"type": "Point", "coordinates": [51.425, 35.8125]}, "properties": {"id": "Record_03", "major": "PHOTO", "date_iso": "2025-07-16", "cloud_pct": null, "primary_metric": 18.6, "qa_valid": true}},
    {"type": "Feature", "geometry": {"type": "Point", "coordinates": [51.35, 35.7]}, "properties": {"id": "Record_04", "major": "GEODESY", "date_iso": "2025-07-16", "cloud_pct": null, "primary_metric": 1420.55, "qa_valid": true}},
    {"type": "Feature", "geometry": {"type": "Point", "coordinates": [51.35, 35.65]}, "properties": {"id": "Record_05", "major": "GIS", "date_iso": "2025-07-17", "cloud_pct": 0.0, "primary_metric": 7.42, "qa_valid": true}},
    {"type": "Feature", "geometry": {"type": "Point", "coordinates": [51.55, 35.85]}, "properties": {"id": "Record_06", "major": "RS", "date_iso": "2025-07-18", "cloud_pct": null, "primary_metric": 0.21, "qa_valid": false}}
  ]
}
'''

data = json.loads(llm_geojson_str)
feats = data["features"]

# 1. Feature Count Checksum
assert len(feats) == 6, f"Expected 6 features, got {len(feats)}"

# 2. Record_03 Exact DMS Conversion Check (35°48'45"N, 51°25'30"E -> [51.425, 35.8125])
r3_coords = feats[2]["geometry"]["coordinates"]
assert round(r3_coords[0], 4) == 51.4250 and round(r3_coords[1], 4) == 35.8125, f"DMS Math Error on Record_03! Got {r3_coords}"

# 3. Record_04 Reversed Coordinate Trap (Must be [Lon=51.35, Lat=35.70], NOT [35.70, 51.35])
r4_coords = feats[3]["geometry"]["coordinates"]
assert 51.0 < r4_coords[0] < 52.0 and 35.0 < r4_coords[1] < 36.0, f"CRS AXIS TRAP FAILED on Record_04! Got {r4_coords}"

# 4. Record_02 NoData Trap (-9999 must become None/null)
assert feats[1]["properties"]["primary_metric"] is None, "NODATA TRAP FAILED: -9999 was not converted to null!"

# 5. Record_06 Physical Bounds Trap (145.0% cloud cover must become None/null)
assert feats[5]["properties"]["cloud_pct"] is None, "PHYSICAL BOUNDS TRAP FAILED: 145% cloud cover was kept!"

print("\n[+] ALL 5 GEOSPATIAL & PHYSICAL CHECKS PASSED! Rendering interactive map...")

# Save valid GeoJSON and render on an interactive satellite basemap
with open("lab2_verified.geojson", "w") as f:
    json.dump(data, f, indent=2)

m = leafmap.Map(center=[35.75, 51.42], zoom=10)
m.add_basemap("Esri.WorldImagery")
m.add_geojson("lab2_verified.geojson", layer_name="Verified Multi-Sensor Campaign")
m

=== 1. TIKTOKENIZER COORDINATE BLOAT EXPERIMENT ===
Verbose DMS String Tokens : 35 tokens
Compact Decimal JSON Tokens: 22 tokens

=== 2. PROMPT GOLF SCORE: 167 Tokens (Target Par <= 135) ===


AssertionError: Over Par! Reduce your prompt by 32 tokens.